# Taller 4

# Librerías

In [9]:
import requests
import pandas as pd
from datetime import datetime

print('Librerías cargadas correctamente')

Librerías cargadas correctamente


# Ciudades a analizar

In [10]:
ciudades = {
    'Bogotá':      {'lat': 4.7110,  'lon': -74.0721},
    'Medellín':    {'lat': 6.2442,  'lon': -75.5812},
    'Cali':        {'lat': 3.4516,  'lon': -76.5320},
    'Barranquilla':{'lat': 10.9639, 'lon': -74.7964},
    'Cartagena':   {'lat': 10.3910, 'lon': -75.4794},
}

print(f'Se consultarán {len(ciudades)} ciudades')

Se consultarán 5 ciudades


## API: Open-Meteo entrega pronóstico diario de temperatura máxima/mínima, precipitación, viento y un código de clima (`weathercode`) para los próximos días.

In [11]:
def consultar_clima(nombre_ciudad, lat, lon):
    """Consulta el pronóstico de 7 días para una ciudad y devuelve el JSON crudo."""
    url = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude': lat,
        'longitude': lon,
        'daily': 'temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max,weathercode',
        'timezone': 'auto',
        'forecast_days': 7,
    }
    respuesta = requests.get(url, params=params, timeout=10)
    respuesta.raise_for_status()  # lanza error si la consulta falla
    return respuesta.json()

# Prueba rápida con una sola ciudad
ejemplo = consultar_clima('Bogotá', ciudades['Bogotá']['lat'], ciudades['Bogotá']['lon'])
ejemplo.keys()

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])

# Consultar todas las ciudades y convertir JSON → DataFrame

In [12]:
filas = []

for nombre, coords in ciudades.items():
    data = consultar_clima(nombre, coords['lat'], coords['lon'])
    diario = data['daily']

    # Cada ciudad devuelve 7 fechas; iteramos sobre cada una
    for i, fecha in enumerate(diario['time']):
        filas.append({
            'ciudad': nombre,
            'fecha': fecha,
            'temp_max': diario['temperature_2m_max'][i],
            'temp_min': diario['temperature_2m_min'][i],
            'precipitacion_mm': diario['precipitation_sum'][i],
            'viento_max_kmh': diario['windspeed_10m_max'][i],
            'weathercode': diario['weathercode'][i],
        })

df_raw = pd.DataFrame(filas)
df_raw.head(10)

,ciudad,fecha,temp_max,temp_min,precipitacion_mm,viento_max_kmh,weathercode
0,Bogotá,2026-08-31,20.6,12.1,1.7,17.7,51
1,Bogotá,2026-09-01,21.6,10.8,1.0,13.8,51
2,Bogotá,2026-09-02,20.4,10.7,1.5,17.9,51
3,Bogotá,2026-09-03,20.5,10.4,1.6,15.2,51
4,Bogotá,2026-09-04,20.4,11.4,0.9,14.2,51
5,Bogotá,2026-09-05,20.8,10.9,0.9,14.9,51
6,Bogotá,2026-09-06,20.9,10.1,2.4,12.5,51
7,Medellín,2026-08-31,30.1,17.4,2.7,6.1,53
8,Medellín,2026-09-01,27.8,17.6,4.3,7.1,80
9,Medellín,2026-09-02,29.0,17.2,3.2,11.3,80


# Limpieza y transformación de datos

In [13]:
# Diccionario de traducción de códigos WMO de clima (Open-Meteo usa el estándar WMO)
codigos_clima = {
    0: 'Despejado', 1: 'Mayormente despejado', 2: 'Parcialmente nublado', 3: 'Nublado',
    45: 'Niebla', 48: 'Niebla con escarcha',
    51: 'Llovizna ligera', 53: 'Llovizna moderada', 55: 'Llovizna densa',
    61: 'Lluvia ligera', 63: 'Lluvia moderada', 65: 'Lluvia fuerte',
    80: 'Chubascos ligeros', 81: 'Chubascos moderados', 82: 'Chubascos fuertes',
    95: 'Tormenta', 96: 'Tormenta con granizo leve', 99: 'Tormenta con granizo fuerte',
}

df = df_raw.copy()

# Conversión de fecha
df['fecha'] = pd.to_datetime(df['fecha'])
df['dia_semana'] = df['fecha'].dt.day_name(locale='es_ES') if False else df['fecha'].dt.day_name()

# Traducción del código de clima (si no existe en el diccionario, se marca 'Sin dato')
df['condicion_clima'] = df['weathercode'].map(codigos_clima).fillna('Sin dato')

# Columna derivada: rango térmico del día
df['rango_termico'] = (df['temp_max'] - df['temp_min']).round(1)

# Redondeo de valores numéricos para presentación limpia
for col in ['temp_max', 'temp_min', 'precipitacion_mm', 'viento_max_kmh']:
    df[col] = df[col].round(1)

# Manejo de nulos (si alguna ciudad no reporta un valor puntual)
df = df.dropna(subset=['temp_max', 'temp_min'])

# Reordenar columnas para el dataset final
df_final = df[['ciudad', 'fecha', 'dia_semana', 'condicion_clima',
               'temp_max', 'temp_min', 'rango_termico',
               'precipitacion_mm', 'viento_max_kmh']]

df_final.head(10)

,ciudad,fecha,dia_semana,condicion_clima,temp_max,temp_min,rango_termico,precipitacion_mm,viento_max_kmh
0,Bogotá,2026-08-31,Monday,Llovizna ligera,20.6,12.1,8.5,1.7,17.7
1,Bogotá,2026-09-01,Tuesday,Llovizna ligera,21.6,10.8,10.8,1.0,13.8
2,Bogotá,2026-09-02,Wednesday,Llovizna ligera,20.4,10.7,9.7,1.5,17.9
3,Bogotá,2026-09-03,Thursday,Llovizna ligera,20.5,10.4,10.1,1.6,15.2
4,Bogotá,2026-09-04,Friday,Llovizna ligera,20.4,11.4,9.0,0.9,14.2
5,Bogotá,2026-09-05,Saturday,Llovizna ligera,20.8,10.9,9.9,0.9,14.9
6,Bogotá,2026-09-06,Sunday,Llovizna ligera,20.9,10.1,10.8,2.4,12.5
7,Medellín,2026-08-31,Monday,Llovizna moderada,30.1,17.4,12.7,2.7,6.1
8,Medellín,2026-09-01,Tuesday,Chubascos ligeros,27.8,17.6,10.2,4.3,7.1
9,Medellín,2026-09-02,Wednesday,Chubascos ligeros,29.0,17.2,11.8,3.2,11.3


# Validación del dataset

In [14]:
print('Dimensiones:', df_final.shape)
print('\nTipos de dato:')
print(df_final.dtypes)
print('\nValores nulos por columna:')
print(df_final.isnull().sum())
print('\nResumen estadístico:')
df_final.describe()

Dimensiones: (35, 9)

Tipos de dato:
ciudad                         str
fecha               datetime64[us]
dia_semana                     str
condicion_clima                str
temp_max                   float64
temp_min                   float64
rango_termico              float64
precipitacion_mm           float64
viento_max_kmh             float64
dtype: object

Valores nulos por columna:
ciudad              0
fecha               0
dia_semana          0
condicion_clima     0
temp_max            0
temp_min            0
rango_termico       0
precipitacion_mm    0
viento_max_kmh      0
dtype: int64

Resumen estadístico:


,fecha,temp_max,temp_min,rango_termico,precipitacion_mm,viento_max_kmh
count,35,35.000000,35.000000,35.000000,35.000000,35.000000
mean,2026-09-03 00:00:00,28.091429,19.931429,8.160000,5.842857,11.417143
min,2026-08-31 00:00:00,20.400000,10.100000,3.500000,0.900000,4.400000
25%,2026-09-01 00:00:00,27.850000,17.300000,5.850000,1.650000,8.350000
50%,2026-09-03 00:00:00,29.300000,19.900000,8.200000,3.500000,11.400000
75%,2026-09-05 00:00:00,30.300000,25.350000,10.150000,7.900000,13.850000
max,2026-09-06 00:00:00,32.600000,26.600000,12.700000,28.200000,18.200000
std,NaN,3.980568,5.624410,2.602962,5.975356,3.909226


# Exportar a CSV

In [15]:
nombre_archivo = 'datos_clima_colombia.csv'
df_final.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')
print(f'Archivo exportado correctamente: {nombre_archivo}')

Archivo exportado correctamente: datos_clima_colombia.csv
